<a href="https://colab.research.google.com/github/oxedanda/pml_final_project/blob/main/notebooks/03_forecast_simulator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Forecast simulator: historical baseline + experimental future forecast

This notebook consolidates the former simulator notebooks into one clearer workflow.

It has two distinct evaluation blocks:

1. **Fixed future test (2023-2025):** models are selected on rolling validation from 2018-2022 and then tested on the untouched years 2023-2025. This is the strict baseline comparison.
2. **One-step-ahead test (2023-2025):** each campaign can use lagged production from the immediately previous campaign. This is closer to an operational next-campaign forecast.

The final section uses the selected lagged model to produce an experimental 2026/27 forecast and a vineyard-area scenario simulator.

In [ ]:
# Colab/local setup
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/oxedanda/pml_final_project.git"
REPO_DIR = Path("/content/pml_final_project")

if "google.colab" in sys.modules:
    if not (REPO_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
else:
    cwd = Path.cwd()
    if cwd.name == "notebooks":
        os.chdir(cwd.parent)

ROOT = Path.cwd()
print(f"Working directory: {ROOT}")

In [ ]:
import pandas as pd
from IPython.display import Image, display

from src.evaluate_models import evaluate, save_outputs as save_baseline_outputs
from src.future_forecast import (
    evaluate_forecasters,
    forecast_2026,
    load_history,
    save_outputs as save_future_outputs,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

## 1. Data used by the simulator

The modelling dataset joins IVV wine production by viticultural region with vineyard area by region and campaign start year.

In [ ]:
history = load_history()
print(f"Rows: {len(history):,}")
print(f"Regions: {history['region'].nunique()}")
print(f"Years: {history['year_start'].min()}-{history['year_start'].max()}")
display(history.head())

## 2. Fixed future test: strict baseline comparison

This block reproduces the main project evaluation. The validation years are 2018-2022, and the final test years 2023-2025 are kept as a fixed future period. No information from 2023-2025 is used when selecting the model.

This answers the question: **how well can models trained up to 2022 predict the future block 2023-2025?**

In [ ]:
baseline_metrics, baseline_predictions = evaluate()
save_baseline_outputs(baseline_metrics, baseline_predictions)

baseline_validation = baseline_metrics[baseline_metrics["split"] == "rolling_validation_2018_2022"].sort_values("MAE")
baseline_test = baseline_metrics[baseline_metrics["split"] == "test_2023_2025"].sort_values("MAE")

print("Rolling validation, 2018-2022")
display(baseline_validation[["model", "MAE", "RMSE", "R2", "selected_on_validation"]])

print("Final fixed test, 2023-2025")
display(baseline_test[["model", "MAE", "RMSE", "R2", "selected_on_validation"]])

display(Image(filename="outputs/figures/model_comparison_mae.png", width=850))

**Interpretation.** The fixed test is deliberately strict. In the current results, the simple persistence baseline is stronger than the selected machine-learning model on 2023-2025. This does not mean the modelling failed; it shows that recent regional production is a very strong benchmark and that region/year/vineyard area alone do not fully capture annual shocks.

## 3. One-step-ahead extension with lagged production

This block adds lagged production features: previous campaign production, two-campaign lag, recent three-campaign mean, and recent production change. These features are shifted within each region, so the target campaign is never used as an input.

This answers a different question: **how well can we forecast the next campaign when the immediately previous campaign is already known?**

In [ ]:
future_evaluation, future_forecast = save_future_outputs()

future_metrics = future_evaluation.metrics.copy()
future_validation = future_metrics[future_metrics["split"] == "rolling_validation_2018_2022"].sort_values("MAE")
one_step_test = future_metrics[future_metrics["split"] == "test_2023_2025"].sort_values("MAE")

print("Selected one-step-ahead model:", future_evaluation.selected_model)
print(f"Empirical 90% interval half-width: {future_evaluation.interval_half_width:,.0f} hl")

print("Rolling validation with lags, 2018-2022")
display(future_validation[["model", "MAE", "RMSE", "R2", "selected_on_validation"]])

print("One-step-ahead test with lags, 2023-2025")
display(one_step_test[["model", "MAE", "RMSE", "R2", "selected_on_validation"]])

The one-step-ahead result should not be compared directly with the fixed future test as if they used the same information. The one-step-ahead test is allowed to use the previous campaign, so it is operationally later and usually easier.

## 4. Experimental 2026/27 forecast

The selected lagged model is refitted using the available history and then used to forecast the next campaign. The default scenario keeps the latest observed vineyard area for each region.

### Explicit prediction step

The notebook calls `forecast_2026(future_evaluation)` below. Inside that function, the selected model is refitted on the supervised historical data and the actual prediction is computed with:

```python
model.fit(supervised[FEATURES], supervised[TARGET])
prediction = model.predict(future[FEATURES])
```

The resulting table is shown as `future_forecast` and saved to `outputs/tables/forecast_2026_27.csv`.


In [ ]:
future_forecast = forecast_2026(future_evaluation)
display(future_forecast)
future_forecast.to_csv("outputs/tables/forecast_2026_27.csv", index=False)

## 5. Vineyard-area scenario simulator

Changing vineyard area here creates a **scenario**, not a causal estimate. The model shows how the forecast changes if a different area is supplied, but it does not prove that changing vineyard area would directly cause that production change.

In [ ]:
def simulate_region_area(region, vineyard_area_ha):
    scenario = forecast_2026(
        future_evaluation,
        area_overrides={region: float(vineyard_area_ha)},
    )
    return scenario[scenario["region"] == region]

example_region = future_forecast["region"].iloc[0]
example_area = float(
    future_forecast.loc[
        future_forecast["region"] == example_region,
        "vineyard_area_ha",
    ].iloc[0]
)

print(f"Example scenario for {example_region} with {example_area:,.2f} ha")
display(simulate_region_area(example_region, example_area))


## 6. Limits to state clearly

- The fixed future test and the one-step-ahead test answer different forecasting questions.
- The empirical 90% interval is based on observed validation errors; it is not a formal statistical confidence interval.
- Vineyard-area changes in the simulator are scenarios, not causal claims.
- Better future forecasts would require weather, drought, disease, management, or remote-sensing predictors known before harvest.